# Lab 6: Training, Tuning, and Scaling k-Nearest Neighbors

In Lab 5, you built 1-nearest-neighbor and k-nearest-neighbors classifiers from scratch. In this lab, you will use **scikit-learn** to carry out the same algorithm, then study three questions that arise when we build a real classifier:

1. How does performance change when the model gets more training data?
2. How should we choose the hyperparameter $k$ without using the test set?
3. What happens when the features used to measure distance have very different scales?

You will answer the first two questions with the Spotify data from Lab 5, repeat the analysis with steel-plate faults, and then fix the steel classifier using a `StandardScaler` inside a scikit-learn pipeline.

You should complete this entire lab so that all tests pass.


In [ ]:
in_colab = "google.colab" in str(get_ipython())
if in_colab:
    !pip install otter-grader==6.1.6

from pathlib import Path
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

path = 'labs/lab06/build/student'
if in_colab and not Path('data').exists():
    !wget -q -O /content/course.zip https://github.com/dsc-courses/cosmos-ml-cluster-2026/archive/refs/heads/main.zip
    with zipfile.ZipFile('/content/course.zip') as course_zip:
        archive_prefix = f'cosmos-ml-cluster-2026-main/{path}/'
        members = [name for name in course_zip.namelist() if name.startswith(archive_prefix)]
        course_zip.extractall('/content/course-assets', members)
    source_path = Path('/content/course-assets') / archive_prefix
    shutil.copytree(source_path / 'data', 'data', dirs_exist_ok=True)
    shutil.copytree(source_path / 'tests', 'tests', dirs_exist_ok=True)

import otter
grader = otter.Notebook()

plt.style.use('seaborn-v0_8-colorblind')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. k-NN with scikit-learn 🎵

We will begin with the same Spotify data and seven numerical features used in Lab 5. Recall that the target is the song's genre: classical, hip-hop, or rock.


In [ ]:
spotify = pd.read_csv('data/spotify-three-genres.csv').rename(columns={'track_genre': 'genre'})
spotify_features = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence'
]

X_spotify = spotify[spotify_features]
y_spotify = spotify['genre']
spotify.head()


### Recreating 5-NN

A scikit-learn model follows a common pattern:

1. Create the model, choosing its hyperparameters.
2. Call `.fit(X_train, y_train)` to train it.
3. Call `.predict(X_new)` or `.score(X_new, y_new)` to use it.

For a classifier, `.score` returns the proportion of correct predictions—its accuracy.

The next cell recreates Lab 5's fixed train/test split so that we can compare implementations.


In [ ]:
lab05_test_indices = spotify.sample(frac=0.2, random_state=42).index
X_lab05_train = X_spotify.drop(index=lab05_test_indices)
X_lab05_test = X_spotify.loc[lab05_test_indices]
y_lab05_train = y_spotify.drop(index=lab05_test_indices)
y_lab05_test = y_spotify.loc[lab05_test_indices]


**Question 1.1.** Create a `KNeighborsClassifier` named `spotify_5nn` with 5 neighbors. Fit it using `X_lab05_train` and `y_lab05_train`. Then store its predictions for `X_lab05_test` in `sklearn_predictions` and its accuracy on that test set in `sklearn_accuracy`.

Your accuracy should match the accuracy of the 5-NN classifier you built from scratch in Lab 5.


In [ ]:
...

sklearn_accuracy


In [ ]:
grader.check("q1_1")

### Training, validation, and test sets

Lab 5 compared several values of $k$ using the test set. That was useful while learning the algorithm, but it creates a problem: if we use test performance to choose $k$, then the test set has influenced our model.

We will instead divide the data into three sets:

| Set | Purpose |
| --- | --- |
| **Training** | Fit the model. |
| **Validation** | Compare choices such as different values of $k$. |
| **Test** | Evaluate the final procedure once, after all choices have been made. |

We want a 60% training, 20% validation, and 20% test split. We can get this with two calls to `train_test_split`:

1. Split the full data into 80% development data and 20% test data.
2. Split the development data into 75% training and 25% validation data. Since 25% of 80% is 20%, the final proportions are 60/20/20.

Setting `stratify=y` keeps the proportions of the three genres similar in every split.


**Question 1.2.** Split `X_spotify` and `y_spotify` into training, validation, and test sets using the procedure above.

- Use `random_state=0` and `stratify=y_spotify` in the first split.
- Use `random_state=0` and stratify using the development labels in the second split.
- Use `test_size=0.2` for the first split and `test_size=0.25` for the second.

Name the first split `X_spotify_dev`, `X_spotify_test`, `y_spotify_dev`, and `y_spotify_test`. Name the second split `X_spotify_train`, `X_spotify_valid`, `y_spotify_train`, and `y_spotify_valid`.


In [ ]:
...

print(f'Training rows:   {len(X_spotify_train):,}')
print(f'Validation rows: {len(X_spotify_valid):,}')
print(f'Test rows:       {len(X_spotify_test):,}')


In [ ]:
grader.check("q1_2")

### Measuring error

**Classification error** is the proportion of incorrect predictions. Since accuracy is the proportion correct,

$$\text{error} = 1 - \text{accuracy}.$$

**Training error** measures the examples used to fit a model. **Validation error** measures separate examples that were not used during fitting.


**Question 1.3.** Complete the function `train_valid_errors`. It takes an already-fitted model and returns a Series containing its training and validation errors. The Series must have the index labels `'train_error'` and `'valid_error'`, in that order.


In [ ]:
def train_valid_errors(model, X_train, y_train, X_valid, y_valid):
    """Return the training and validation classification errors of a fitted model."""
    ...

spotify_5nn_split = KNeighborsClassifier(n_neighbors=5).fit(X_spotify_train, y_spotify_train)
train_valid_errors(
    spotify_5nn_split,
    X_spotify_train,
    y_spotify_train,
    X_spotify_valid,
    y_spotify_valid,
)


In [ ]:
grader.check("q1_3")

### What happens when we get more training data?

We will train 5-NN using increasingly large subsets of the training set. Every model will be evaluated on the same validation set. The test set remains untouched.

The fixed random ordering below makes the subsets nested: the 100-row subset contains the first 50 rows, the 200-row subset contains the first 100, and so on.


In [ ]:
spotify_training_sizes = [50, 100, 200, 400, 800, 1200, len(X_spotify_train)]
spotify_training_order = X_spotify_train.sample(frac=1, random_state=0).index


**Question 1.4.** For every size in `spotify_training_sizes`:

1. Select the first `size` rows from `spotify_training_order`.
2. Fit a new 5-NN classifier using only those rows.
3. Compute its training error on that subset and its validation error on the full validation set.

Store the results in a DataFrame named `spotify_errors_by_size`. Its index should be named `'num_training_points'`, and its two columns should be `'train_error'` and `'valid_error'`. Then make a line plot of both errors with markers.


In [ ]:
...

spotify_errors_by_size


In [ ]:
grader.check("q1_4")

**Pause and interpret the graph.** As the number of training songs grows, what generally happens to validation error? Why is the curve not perfectly smooth? Why have we still not used `X_spotify_test`?


### What happens as $k$ increases?

The number of neighbors is a **hyperparameter**: we choose it before fitting the model. Small values of $k$ create a flexible rule that can be strongly affected by individual training examples. Very large values make many distant songs vote, producing a rule that may be too simple.

The values below cover both extremes. They are unevenly spaced, so we will use a logarithmic horizontal axis when plotting them.


In [ ]:
k_values = [1, 5, 15, 25, 101, 401, 801]


**Question 1.5.** Fit one Spotify k-NN classifier for every value in `k_values`, always using the entire Spotify training set. Store the training and validation errors in `spotify_errors_by_k`, indexed by `k`, with columns `'train_error'` and `'valid_error'`.

Make a line plot of both errors with markers. Set the horizontal axis to logarithmic scale using `plt.xscale('log')`.


In [ ]:
...

spotify_errors_by_k


In [ ]:
grader.check("q1_5")

The validation error first decreases and then increases. We choose the value of $k$ at the bottom of this **validation-error valley**. The training error, however, generally increases with $k$. A model can fit its training data extremely well and still be worse on unseen data.


**Question 1.6.** Set `best_spotify_k` to the value of $k$ with the smallest validation error. If there is a tie, `.idxmin()` will choose the first tied value.

Now that $k$ has been chosen, combine the training and validation sets, fit a new classifier named `final_spotify_model` using all of that development data, and store its error on the untouched test set in `spotify_test_error`.


In [ ]:
...

best_spotify_k, spotify_test_error


In [ ]:
grader.check("q1_6")

## 2. Repeating the process with steel-plate faults 🏭

Each row in the next dataset describes a fault found on the surface of a steel plate. The 24 numerical features are measurements extracted from an image of the fault. The target, `fault_type`, identifies one of seven kinds of faults.

The `id` column identifies a row but is not a physical measurement, so it must not be used as a model feature.


In [ ]:
steel = pd.read_csv('data/steel-plate-faults.csv')
steel.head()


**Question 2.1.** Create `X_steel` using every column except `'id'` and `'fault_type'`, and create `y_steel` using the `'fault_type'` column.

Then create 60/20/20 training, validation, and test sets using the same two-step procedure as Question 1.2. Use `random_state=0` and stratify both splits. Follow the same naming pattern as the Spotify variables, replacing `spotify` with `steel` (for example, `X_steel_dev` and `X_steel_train`).


In [ ]:
...

print(f'Training rows:   {len(X_steel_train):,}')
print(f'Validation rows: {len(X_steel_valid):,}')
print(f'Test rows:       {len(X_steel_test):,}')


In [ ]:
grader.check("q2_1")

In [ ]:
steel_training_sizes = [50, 100, 200, 400, 600, 800, len(X_steel_train)]
steel_training_order = X_steel_train.sample(frac=1, random_state=0).index


**Question 2.2.** Repeat Question 1.4 using the steel data. Train a 5-NN classifier with every size in `steel_training_sizes`, and evaluate every classifier using the same steel validation set.

Store the results in `steel_errors_by_size`, indexed by `'num_training_points'`, with columns `'train_error'` and `'valid_error'`. Make a line plot of the errors with markers.


In [ ]:
...

steel_errors_by_size


In [ ]:
grader.check("q2_2")

**Question 2.3.** Repeat Question 1.5 using the entire steel training set and every value in `k_values`.

Store the results in `steel_errors_by_k`, indexed by `k`, with columns `'train_error'` and `'valid_error'`. Make a line plot with markers and a logarithmic horizontal axis.


In [ ]:
...

steel_errors_by_k


In [ ]:
grader.check("q2_3")

**Question 2.4.** Set `best_raw_steel_k` to the value of $k$ with the smallest validation error, and set `best_raw_steel_valid_error` to that error.

Do **not** evaluate this model on the test set. We have not finished making modeling choices yet.


In [ ]:
...

best_raw_steel_k, best_raw_steel_valid_error


In [ ]:
grader.check("q2_4")

## 3. The feature-scale problem ⚖️

k-NN decides which rows are nearby using distance. This means that the units and scales of the features matter.

Consider two steel features:

- `X_Minimum` is a horizontal pixel coordinate, with values reaching into the thousands.
- `Edges_Index` is a precomputed edge measurement, with values roughly between 0 and 1.

A difference of 100 pixels in `X_Minimum` overwhelms even the largest possible difference in `Edges_Index`. In the distance calculation, `X_Minimum` therefore receives much more influence—not because we decided it was more important, but simply because its numbers are larger.


In [ ]:
two_features = ['X_Minimum', 'Edges_Index']
X_steel_train[two_features].agg(['min', 'max', 'mean', 'std']).T


### Standardization

`StandardScaler` transforms each feature using statistics learned from the training data:

$$\text{standardized value} = \frac{\text{value} - \text{training mean}}{\text{training standard deviation}}.$$

After this transformation, every training feature has mean 0 and standard deviation 1. A one-unit change then means approximately one training standard deviation, regardless of the feature's original units.


**Question 3.1.** Create a `StandardScaler` named `two_feature_scaler`. Use its `.fit_transform` method to standardize the two columns in `X_steel_train[two_features]`.

Store the result in a DataFrame named `steel_two_standardized`. Give it the same index and column names as `X_steel_train[two_features]`.


In [ ]:
...

steel_two_standardized.head()


In [ ]:
grader.check("q3_1")

The plots below use the same physical scale on the horizontal and vertical axes: one unit has the same displayed length in either direction. In the raw plot, the variation in `Edges_Index` is crushed into an almost-flat line. After standardization, both features contribute visible variation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(
    X_steel_train['X_Minimum'],
    X_steel_train['Edges_Index'],
    alpha=0.35,
    s=18,
)
raw_min = X_steel_train[two_features].min().min()
raw_max = X_steel_train[two_features].max().max()
raw_padding = 0.05 * (raw_max - raw_min)
axes[0].set_xlim(raw_min - raw_padding, raw_max + raw_padding)
axes[0].set_ylim(raw_min - raw_padding, raw_max + raw_padding)
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_title('Raw features (equal unit lengths)')
axes[0].set_xlabel('X_Minimum')
axes[0].set_ylabel('Edges_Index')

axes[1].scatter(
    steel_two_standardized['X_Minimum'],
    steel_two_standardized['Edges_Index'],
    alpha=0.35,
    s=18,
)
standardized_limit = np.abs(steel_two_standardized.to_numpy()).max() * 1.05
axes[1].set_xlim(-standardized_limit, standardized_limit)
axes[1].set_ylim(-standardized_limit, standardized_limit)
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_title('Standardized features (equal unit lengths)')
axes[1].set_xlabel('standardized X_Minimum')
axes[1].set_ylabel('standardized Edges_Index')

plt.tight_layout()


### Putting transformations in a pipeline

We could manually fit a scaler, transform the training data, remember to transform validation and test data using the **same** scaler, and then fit k-NN. A scikit-learn **pipeline** safely combines those steps:

```python
model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)
```

When `.fit` is called, the pipeline fits the scaler using the training data, transforms the training features, and fits k-NN. When `.predict` or `.score` is called, it transforms new features using the already-fitted scaler before passing them to k-NN.

This prevents us from accidentally fitting a separate scaler to validation or test data.


**Question 3.2.** Create a pipeline named `scaled_steel_5nn` containing a `StandardScaler` followed by a 5-NN classifier. Fit it on the steel training data, then store its training and validation errors in `scaled_steel_5nn_errors` using `train_valid_errors`.


In [ ]:
...

scaled_steel_5nn_errors


In [ ]:
grader.check("q3_2")

**Question 3.3.** Tune $k$ again, this time creating and fitting a new `StandardScaler`/k-NN pipeline for every value in `k_values`.

Store the errors in `scaled_steel_errors_by_k`, indexed by `k`, with columns `'train_error'` and `'valid_error'`. Set `best_scaled_steel_k` to the value with the smallest validation error.

Finally, create `steel_scaling_comparison`, a DataFrame with columns `'raw features'` and `'standardized features'` containing the two sets of validation errors. Make a line plot of this comparison with markers and a logarithmic horizontal axis.


In [ ]:
...

best_scaled_steel_k, scaled_steel_errors_by_k


In [ ]:
grader.check("q3_3")

Standardization substantially lowers validation error because all 24 features can now influence distance on comparable scales. Notice that we made this comparison using validation data, not test data.


**Question 3.4.** We have finished choosing the preprocessing and the value of $k$. Combine the steel training and validation sets.

Create `final_steel_model`, a new pipeline containing `StandardScaler` and a k-NN classifier using `best_scaled_steel_k`. Fit it using the combined training and validation data. Finally, store its error on the untouched steel test set in `steel_test_error`.


In [ ]:
...

steel_test_error


In [ ]:
grader.check("q3_4")

## Finish Line 🏁

In this lab, you moved from implementing k-NN yourself to using it as part of a complete modeling workflow:

- More training examples generally improve performance on unseen data.
- Training error alone cannot tell us which hyperparameter will generalize best.
- Validation data is used to choose $k$ and preprocessing; test data is reserved for one final evaluation.
- Since k-NN uses distances, features with large numerical scales can dominate its predictions.
- A pipeline fits `StandardScaler` on training data and applies the same transformation before k-NN makes predictions.

Before finishing, discuss these questions with a neighbor:

1. Why did we retrain each final model using both the training and validation sets?
2. Why did standardization help much more for the steel data than for the Spotify data?
3. Why would choosing a new model after seeing its test error make that test error less trustworthy?


In [ ]:
# For your convenience, you can run this cell to run all the tests at once!
grader.check_all()
